# **Pre-Trained Model 2 - MobileNetV2**

As a third pre-trained model we have choosen DenseNet's for its core idea — dense connectivity, where every layer receives feature
maps from all preceding layers, which we believe to be well-suited for our model. The dense connections encourage feature reuse, which helps the network
learn subtle dermoscopic patterns (pigment networks, vascular structures) with
fewer parameters than equivalent ResNet-style architectures.

The three variants of the model will be tested - ....

### Imports

In [ ]:
import os
import keras
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from model_utils import get_callbacks, plot_history, evaluate_model, plot_comparison

### **Data** Configuration

In [ ]:
# path for the new images
base_path = "./data" 
aug_dir = os.path.join(base_path, "HAM10000_augmented")
if not os.path.exists(aug_dir):
    os.makedirs(aug_dir)

In [ ]:
cols = ['image_id', 'dataset', 'lesion_id', 'aug_path', 'dx']

train_df = pd.read_csv('data/train_split.csv')[cols]
val_df   = pd.read_csv('data/val_split.csv')[cols]
test_df  = pd.read_csv('data/test_split.csv')[cols]

train_df = train_df.rename(columns={'cleaned_path': 'image_path'})
val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dataset'] = 'original'
val_df['dataset'] = 'original'
test_df['dataset'] = 'original'

# Add encoded labels
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

for df in [train_df, val_df, test_df]:
    df['dx_encoded'] = df['dx'].map(label2idx)


### **Model** Configuration

In [ ]:
UNFREEZE_FROM_C = -20   # MobileNetV2 is shallower — unfreeze last 20 layers

def build_mobilenet():
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        alpha=1.0
    )
    base.trainable = False

    inputs  = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)
    return keras.Model(inputs, outputs, name="mobilenet_v2")

In [ ]:
# Phase 1: head only 
model_pt2 = build_mobilenet() # Model Pre-trained 2
model_pt2.summary()

model_pt2.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history1_pt2 = model_pt2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_c_phase1.weights.h5",
                            patience_es=6, patience_lr=3)
)

plot_history(history1_pt2, "Model C — MobileNetV2 Phase 1 (head only)")

In [ ]:
# Phase 2: fine-tune top layers 
base_c = model_pt2.layers[1]
base_c.trainable = True
for layer in base_c.layers[:UNFREEZE_FROM_C]:
    layer.trainable = False

model_pt2.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history2_pt2 = model_pt2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_c_best.weights.h5",
                            patience_es=8, patience_lr=4)
)

plot_history(history2_pt2, "Model C — MobileNetV2 Phase 2 (fine-tune)")
results_c = evaluate_model(model_pt2, test_ds, test_df, label2idx, "Model C — MobileNetV2")